# Checker-only usage: SMT and Lean

When you already have a specific formal claim and only need symbolic verification — without running the full reasoning loop — you can call `SMTChecker` or `LeanChecker` directly.

Install the package first if you have not already:

```bash
pip install -e .
```

In [ ]:
from agentic_validation.checkers import SMTChecker, LeanChecker
from agentic_validation.schemas import FormalClaim

## 1. Build a `FormalClaim`

A `FormalClaim` pairs a natural-language claim text with a machine-checkable expression and a target checker.

In [ ]:
claim = FormalClaim(
    claim_id="claim-1",
    claim_text="x > 5 implies x + 1 > 6",
    target="smt",
    formal_expression="Implies(GT(Symbol('x', INT), Int(5)), GT(Plus(Symbol('x', INT), Int(1)), Int(6)))",
)
print(claim.model_dump_json(indent=2))

## 2. Run the SMT checker

In [ ]:
smt = SMTChecker()
result = smt.check(claim, assumptions=[], steps=[])
print("Status:", result.status)
print("Message:", result.message)

## 3. Run the Lean checker

The Lean checker requires a Lean 4 toolchain on `PATH`.  Install it with [elan](https://github.com/leanprover/elan) if needed:

```bash
curl https://elan.run -sSf | sh
```

The `target` must be `"lean"` and `formal_expression` must be valid Lean 4 syntax.

In [ ]:
lean_claim = FormalClaim(
    claim_id="claim-lean-1",
    claim_text="x > 5 implies x + 1 > 6",
    target="lean",
    formal_expression="theorem t (x : Int) (h : x > 5) : x + 1 > 6 := by omega",
)

lean = LeanChecker()
lean_result = lean.check(lean_claim, assumptions=[], steps=[])
print("Status:", lean_result.status)
print("Message:", lean_result.message)

## Key fields in `CheckerResult`

| Field | Description |
|---|---|
| `status` | `verified`, `rejected`, or `unknown` |
| `checker_type` | `smt` or `lean` |
| `message` | Human-readable explanation |
| `counterexample` | Counter-model if the claim was falsified (SMT only) |
| `artifacts` | Raw solver output for replay/audit |